# W03 - Data Quality and Issues: Complete Guide

## Overview
Noise and outliers are two of the most important data quality problems in machine learning and data preprocessing. Although both appear unusual inside datasets, they represent fundamentally different phenomena.

- **Noise** refers to corrupted or distorted data caused by random errors.
- **Outliers** are legitimate but unusual observations that significantly differ from the majority of the dataset.

Understanding the distinction is critical because machine learning systems handle them differently.

In [ ]:
# Always start with imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

# Set display options
pd.set_option('display.max_rows', 100)
pd.set_option('display.max_columns', None)

# Set random seed for reproducibility
np.random.seed(42)

print("Libraries successfully imported and environment configured!")

## 1. Introduction to Data Quality: Creating a Baseline

Before we understand how data gets corrupted, let's create a baseline "True" dataset. 
Imagine we are tracking the true temperature in a controlled environment over time.

In [ ]:
# Generate true, clean synthetic data
time_steps = np.arange(0, 100)

# Let's assume the true temperature follows a gentle sine wave pattern
true_temperature = 20 + 5 * np.sin(time_steps / 10.0)

# Create a DataFrame to hold our data
df_env = pd.DataFrame({
    'Time': time_steps,
    'True_Temp': true_temperature
})

print("Preview of true, uncorrupted data:")
print(df_env.head())

# Visualize the true data
plt.figure(figsize=(10, 4))
plt.plot(df_env['Time'], df_env['True_Temp'], color='blue', linewidth=2, label='True Data')
plt.title('Baseline: True Data Before Any Issues')
plt.xlabel('Time')
plt.ylabel('Temperature')
plt.grid(alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

## 2. Understanding Data Noise

Noise refers to unwanted modification in original data values. It is essentially random error or variance introduced into measurements.

Mathematically:
X_observed = X_true + epsilon

where:
- X_true represents the actual value
- epsilon represents random error or variance

Noise is not considered a legitimate data point. It is corrupted information introduced during collection, transmission, storage, or measurement.

In [ ]:
# Simulating random noise (epsilon)
# Random normal distribution centered at 0 with standard deviation of 1.5
epsilon = np.random.normal(loc=0.0, scale=1.5, size=len(df_env))

# Applying the formula: X_observed = X_true + epsilon
df_env['Observed_Temp'] = df_env['True_Temp'] + epsilon

print("Data with added random noise:")
print(df_env[['Time', 'True_Temp', 'Observed_Temp']].head())

# Visualization of Noise
plt.figure(figsize=(10, 4))
plt.plot(df_env['Time'], df_env['True_Temp'], color='blue', linewidth=2, label='True Data (X_true)')
plt.scatter(df_env['Time'], df_env['Observed_Temp'], color='red', alpha=0.6, label='Noisy Observations (X_observed)')
plt.title('Understanding Data Noise: X_observed = X_true + epsilon')
plt.xlabel('Time')
plt.ylabel('Temperature')
plt.grid(alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

## 3. Causes of Noisy Data: Faulty Sensors

Noise can emerge from several real-world operational failures. One of the most common sources is malfunctioning sensors.

Suppose a humidity sensor consistently adds a small systematic error:
Humidity_measured = Humidity_actual + delta

where 'delta' is the sensor error.

In [ ]:
# Simulating a faulty sensor with a systematic bias (delta)
true_humidity = np.array([70, 65, 80, 72, 68, 75, 78, 62])
sensor_error_delta = 4  # The sensor always overestimates by 4%

measured_humidity = true_humidity + sensor_error_delta

df_sensor = pd.DataFrame({
    'Actual_Humidity': true_humidity,
    'Sensor_Output': measured_humidity
})

print("Faulty Sensor Measurements:")
print(df_sensor)
print(f"\nNotice how every reading is exactly {sensor_error_delta}% higher than the actual value.")
print("This systematic noise degrades machine learning performance if not calibrated.")

## 4. Causes of Noisy Data: Human Errors & Transmission

### Human Data Entry Errors
A government worker asks how many people live in a house. 
Actual answer: Members = 3. Recorded answer: Members = 5.
This introduces unintentional, accidental distortion.

### Data Transmission Errors
Noise introduced while transmitting data between systems over a network (signal corruption).

In [ ]:
# 1. Simulating Human Error (Typo during census)
actual_members = np.full(20, 3)  # 20 houses, all actually have 3 members
recorded_members = actual_members.copy()

# Introduce human typos (pressing 5 instead of 3)
typo_indices = np.random.choice(20, 3, replace=False)
recorded_members[typo_indices] = 5

print("Census Data (Human Error Simulation):")
print(f"Actual:   {actual_members}")
print(f"Recorded: {recorded_members}")

# 2. Simulating Transmission Error (Data drop/corruption)
transmitted_signal = np.ones(15) * 100
received_signal = transmitted_signal.copy()

# Network glitch drops the value drastically for two packets
glitch_indices = [5, 12]
received_signal[glitch_indices] = 0 

print("\nTransmission Data (Network Glitch Simulation):")
print(f"Transmitted: {transmitted_signal}")
print(f"Received:    {received_signal}")

## 5. Causes of Noisy Data: Technological Limitations

Sometimes the issue is not malfunction but insufficient hardware precision.
Precision_required = 0.01m, but Precision_available = 1m.
This creates approximation noise.

In [ ]:
# Simulating approximation noise due to technological limits
# We need highly precise measurements
precise_measurements = np.random.uniform(10.0, 15.0, 10)

# The hardware can only record integer values (rounding)
hardware_measurements = np.round(precise_measurements)

df_tech = pd.DataFrame({
    'Required_Precision': precise_measurements,
    'Hardware_Output': hardware_measurements
})

# Calculate the approximation noise introduced
df_tech['Approximation_Noise'] = np.abs(df_tech['Required_Precision'] - df_tech['Hardware_Output'])

print("Technological Limitation (Approximation Noise):")
print(df_tech)
print(f"\nAverage approximation error: {df_tech['Approximation_Noise'].mean():.4f}")

## 6. Understanding Outliers

Outliers are fundamentally different from noise.
An outlier is a legitimate data point containing genuine information, but it is statistically unusual compared to the rest of the dataset.

Formally:
x_i in Dataset, but Distance(x_i, mu) >> Most Other Points

Unlike noisy data, outliers are not corruption. They are real but extreme observations.

In [ ]:
# Generating a dataset representing salaries in a typical company (in Lakhs Per Annum - LPA)
normal_salaries = np.random.normal(loc=6.0, scale=1.5, size=100)

# Adding an outlier (e.g., the CEO's salary: 10 Crore = 1000 LPA)
ceo_salary = np.array([1000.0])

all_salaries = np.concatenate([normal_salaries, ceo_salary])

print(f"Most salaries hover around: {np.mean(normal_salaries):.2f} LPA")
print(f"Maximum standard salary: {np.max(normal_salaries):.2f} LPA")
print(f"The extreme outlier salary: {ceo_salary[0]} LPA")

print("\nThis 1000 LPA is NOT noise. It is a real, legitimate salary. But it is an outlier.")

## 7. Real-World Examples: Fraud Detection

Banks continuously analyze transaction patterns. Most transactions follow normal behavior (e.g., Goa, INR 2000). Suddenly, a suspicious activity occurs (USA, INR 2,00,000).

This transaction becomes an outlier and triggers additional verification (OTP/Fraud checks).

In [ ]:
# Simulating Credit Card Transactions
# Normal transactions around 2000 INR
transactions = np.random.normal(loc=2000, scale=500, size=200)

# Adding a fraudulent outlier transaction
fraud_transaction = np.array([200000])
transactions = np.append(transactions, fraud_transaction)

# Calculate Z-scores to detect the outlier
# Z = (X - Mean) / Std
z_scores = np.abs(stats.zscore(transactions))

df_bank = pd.DataFrame({
    'Amount_INR': transactions,
    'Z_Score': z_scores
})

# Flagging anything with a Z-score > 3 as an anomaly/outlier
df_bank['Is_Outlier'] = df_bank['Z_Score'] > 3

print("Transaction Monitoring System:")
print(df_bank.sort_values(by='Amount_INR', ascending=False).head())
print("\nNotice how the massive transaction is automatically flagged for OTP verification.")

## 8. Visual Understanding of Outliers

Conceptually, normal data lies near the cluster center, while outliers lie far from the cluster.
Let's visualize this dense cluster of normal points versus isolated observations.

In [ ]:
# Visualization of the Bank Transactions
plt.figure(figsize=(12, 5))

# Subplot 1: Scatter plot
plt.subplot(1, 2, 1)
colors = ['red' if x else 'blue' for x in df_bank['Is_Outlier']]
plt.scatter(range(len(df_bank)), df_bank['Amount_INR'], c=colors, alpha=0.6)
plt.title('Scatter Plot: Normal vs Outlier')
plt.xlabel('Transaction ID')
plt.ylabel('Amount (INR)')
plt.grid(alpha=0.3)

# Subplot 2: Boxplot (Classic way to show outliers)
plt.subplot(1, 2, 2)
sns.boxplot(y=df_bank['Amount_INR'], color='lightblue')
plt.title('Boxplot highlighting extreme outlier')
plt.ylabel('Amount (INR)')
plt.grid(alpha=0.3)

plt.tight_layout()
plt.show()

## 9. Noise vs Outliers: The Core Differences

Although both appear abnormal, their meanings differ significantly.

| Aspect | Noise | Outlier |
|---|---|---|
| **Nature** | Corrupted Data | Genuine Data |
| **Cause** | Random Error | Rare Event |
| **Legitimacy**| Invalid | Valid |
| **Goal** | Remove | Analyze carefully |

In [ ]:
# Creating a dataset that contains BOTH noise and outliers to show the difference

x = np.linspace(0, 10, 50)
y_true = 2 * x + 1  # Clean linear relationship

# Add Noise (random small variations everywhere)
y_noisy = y_true + np.random.normal(0, 1.5, size=len(x))

# Add an Outlier (one valid but extreme point, e.g., a massive sudden spike in genuine demand)
y_with_outlier = y_noisy.copy()
y_with_outlier[25] = 40  # Massive spike at the middle point

plt.figure(figsize=(10, 5))
plt.plot(x, y_true, 'g--', label='True Pattern (Hidden)', linewidth=2)
plt.scatter(x, y_noisy, c='blue', alpha=0.5, label='Noisy Variations (Corrupted)')
plt.scatter(x[25], y_with_outlier[25], c='red', s=100, marker='*', label='Outlier (Genuine but Extreme)')

plt.title('Noise vs Outlier in a Single Dataset')
plt.xlabel('X')
plt.ylabel('Y')
plt.legend()
plt.grid(alpha=0.3)
plt.show()

## 10. Why Outliers Matter in Machine Learning

Many machine learning algorithms are highly sensitive to outliers. Outliers can heavily distort mean calculations, regression slopes, and distance metrics.

Let's look at how one outlier ruins the Mean calculation.

In [ ]:
# Recalling our salary dataset
salaries_clean = np.array([5, 6, 7, 5.5, 6.5, 7.5, 6, 5])
salaries_with_outlier = np.append(salaries_clean, [1000]) # 10 Crore outlier

mean_clean = np.mean(salaries_clean)
mean_outlier = np.mean(salaries_with_outlier)

print(f"Mean Salary (Clean Data): {mean_clean:.2f} LPA")
print(f"Mean Salary (With Outlier): {mean_outlier:.2f} LPA")
print("\nConclusion: The mean is NOT robust to outliers. It is completely distorted.")

### Impact on Linear Regression

Algorithms like Linear Regression are highly sensitive because they try to minimize squared errors. A large outlier creates a massive error, pulling the entire regression line towards it.

In [ ]:
# Demonstrating impact on Linear Regression
X = np.array([1, 2, 3, 4, 5, 6, 7, 8, 9, 10])
Y_clean = np.array([2, 4, 5, 4, 5, 7, 8, 9, 9, 11])

Y_outlier = Y_clean.copy()
Y_outlier[8] = 25  # Introducing an outlier at X=9

# Fit lines (degree 1 polynomial = linear regression)
slope_clean, intercept_clean = np.polyfit(X, Y_clean, 1)
slope_out, intercept_out = np.polyfit(X, Y_outlier, 1)

line_clean = slope_clean * X + intercept_clean
line_out = slope_out * X + intercept_out

plt.figure(figsize=(10, 5))
plt.scatter(X, Y_clean, c='blue', label='Clean Data')
plt.scatter(9, 25, c='red', s=100, marker='X', label='Outlier Point')

plt.plot(X, line_clean, 'g--', linewidth=2, label=f'Regression Line (Clean)')
plt.plot(X, line_out, 'r-', linewidth=2, label=f'Regression Line (Distorted)')

plt.title('Impact of a Single Outlier on Linear Regression')
plt.legend()
plt.grid(alpha=0.3)
plt.show()

print("The red line is drastically pulled upward by just ONE outlier point.")

## 11. Order of Handling: Noise Before Outliers

A major engineering principle in preprocessing pipelines is:
Handle Noise First => Detect Outliers Later

Reason: Noise itself may appear like an outlier. If noisy points are not removed first, the system may incorrectly classify corrupted observations as genuine anomalies.

In [ ]:
# Building a 2-Step Processing Pipeline

# 1. Create a signal with BOTH noise and one true anomaly (outlier)
np.random.seed(10)
base_signal = np.ones(50) * 50

# Add heavy random noise
noisy_signal = base_signal + np.random.normal(0, 5, 50)

# Add one true outlier
noisy_signal[25] = 90 

# STEP 1: Handle Noise First (Smoothing using a rolling median to remove random variance)
smoothed_signal = pd.Series(noisy_signal).rolling(window=3, center=True).median().fillna(method='bfill').fillna(method='ffill').values

# STEP 2: Detect Outliers on the smoothed data (using Z-score)
z_scores_smoothed = np.abs(stats.zscore(smoothed_signal))
detected_outliers = z_scores_smoothed > 3

plt.figure(figsize=(12, 5))
plt.plot(noisy_signal, alpha=0.5, color='gray', label='Raw Noisy Signal')
plt.plot(smoothed_signal, color='blue', linewidth=2, label='Smoothed Signal (Noise Removed)')

# Highlight detected outliers
outlier_indices = np.where(detected_outliers)[0]
plt.scatter(outlier_indices, smoothed_signal[outlier_indices], color='red', s=100, zorder=5, label='Detected Outlier')

plt.title('Pipeline: Noise Removal followed by Outlier Detection')
plt.legend()
plt.grid(alpha=0.3)
plt.show()

## 12. Practice Exercises

### Exercise 1: Identify and fix simple transmission noise
You receive a stream of percentage data (0-100). Any value outside this range is a transmission error (noise) and should be capped.

In [ ]:
# Exercise 1 Setup
received_percentages = np.array([45, 50, 55, 999, 60, -50, 65, 70])
print(f"Raw Data: {received_percentages}")

# Solution
def clean_transmission_noise(data):
    # Cap values at 0 and 100
    cleaned = np.clip(data, 0, 100)
    return cleaned

fixed_data = clean_transmission_noise(received_percentages)
print(f"Fixed Data: {fixed_data}")

### Exercise 2: IQR-based Outlier Detection
The Interquartile Range (IQR) method is robust against noise when finding outliers. Implement it.

In [ ]:
# Exercise 2 Setup
test_data = pd.Series([10, 12, 11, 14, 13, 15, 12, 100, 11, 14, -80, 13])

# Solution
def detect_outliers_iqr(series):
    Q1 = series.quantile(0.25)
    Q3 = series.quantile(0.75)
    IQR = Q3 - Q1
    
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    
    # Return a boolean mask of outliers
    outliers_mask = (series < lower_bound) | (series > upper_bound)
    return outliers_mask, lower_bound, upper_bound

mask, lb, ub = detect_outliers_iqr(test_data)
print(f"Lower Bound: {lb:.2f}, Upper Bound: {ub:.2f}")
print("\nIdentified Outliers:")
print(test_data[mask].values)

## 13. Summary and Key Takeaways

- **Distinct Phenomena**: Noise represents data corruption (errors), while outliers represent legitimate but extreme variations.
- **Causes of Noise**: Faulty sensors, human entry typos, transmission packet drops, and technological precision limits.
- **Real-world Outliers**: Credit card fraud, unusual login locations, sudden stock market spikes.
- **ML Sensitivity**: Outliers can severely distort models like Linear Regression and metrics like the Mean.
- **Golden Rule**: Always handle Noise FIRST, then detect Outliers LATER. Corrupted data can trigger false anomaly alerts.

In [ ]:
# Final Visualization Gallery summarizing the concepts
fig, axs = plt.subplots(1, 3, figsize=(16, 5))

# 1. Noise Plot
x_vals = np.linspace(0, 10, 20)
y_clean = 2 * x_vals
y_noise = y_clean + np.random.normal(0, 2, 20)
axs[0].plot(x_vals, y_clean, 'k--', label='True')
axs[0].scatter(x_vals, y_noise, c='blue', alpha=0.6, label='Noisy')
axs[0].set_title('1. Data Noise (Random Errors)')
axs[0].legend()
axs[0].grid(alpha=0.3)

# 2. Outlier Plot
y_out = y_clean.copy()
y_out[10] = 35
axs[1].scatter(x_vals, y_clean, c='green', label='Normal Data')
axs[1].scatter(x_vals[10], y_out[10], c='red', s=100, marker='*', label='Outlier')
axs[1].set_title('2. Outlier (Extreme Valid Point)')
axs[1].legend()
axs[1].grid(alpha=0.3)

# 3. Distribution View (Boxplot)
dist_data = np.append(np.random.normal(10, 2, 100), [25, 28])
sns.boxplot(y=dist_data, ax=axs[2], color='lightpurple' if 'lightpurple' in sns.colors.crayons else 'plum')
axs[2].set_title('3. Statistical View of Outliers')
axs[2].grid(alpha=0.3)

plt.tight_layout()
plt.show()

print("Module Complete: W03 - Data Quality and Issues Successfully Rendered.")